# Crop Yield Prediction - Multivariate Regression

**Mission:**
**Use Case:**
**Yield (tonnes per hetare)**
**Dataset:**
**Target:** `yield`

In [ ]:
# 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.utils import shuffle

sns.set_style("whitegrid")

# 2. Loading the data  

In [9]:
DATA_PATH = "hvstat_africa_data_v1.0.csv"

df = pd.read_csv(DATA_PATH)

print("Raw shape:", df.shape)
df.head()

df.info()

Raw shape: (203125, 16)
<class 'pandas.DataFrame'>
RangeIndex: 203125 entries, 0 to 203124
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   fnid                    203125 non-null  str    
 1   country                 203125 non-null  str    
 2   country_code            203125 non-null  str    
 3   admin_1                 203125 non-null  str    
 4   admin_2                 203125 non-null  str    
 5   product                 203125 non-null  str    
 6   season_name             203125 non-null  str    
 7   planting_year           203125 non-null  int64  
 8   planting_month          203125 non-null  int64  
 9   harvest_year            203125 non-null  int64  
 10  harvest_month           203125 non-null  int64  
 11  crop_production_system  203125 non-null  str    
 12  qc_flag                 203125 non-null  int64  
 13  area                    190428 non-null  float64
 14  product

# 3. Cleaning the data

In [12]:
print("Rows before cleaning:", len(df))

# 1. missing values
df = df.dropna(subset=["yield", "area"])
print("After dropping missing yield/area:", len(df))

# 2. zero or negative values
df = df[(df["yield"] > 0) & (df["area"] > 0)]
print("After dropping zero/negative:", len(df))

# 3. extreme yields
df = df[(df["yield"] <= 50)]
print("After removing yield > 50:", len(df))

# 4. old years
df = df[df["harvest_year"] >= 1995].copy()
print("After keeping 1995 onwards:", len(df))

Rows before cleaning: 184990
After dropping missing yield/area: 184990
After dropping zero/negative: 184990
After removing yield > 50: 184990
After keeping 1995 onwards: 160005


# Group rare categories

In [13]:
def keep_top(column, n):
    # keep the n most common values, replace everything with "Other"
    top_values = column.value_counts().nlargest(n).index
    return column.where(column.isin(top_values), "Other")

df["season_name"] = keep_top(df["season_name"], 9)
df["crop_production_system"] = keep_top(df["crop_production_system"], 5)

print(df["season_name"].value_counts())
print()
print(df["crop_production_system"].value_counts())

season_name
Main      68360
Annual    41588
Meher     13570
Other     13359
Wet        6932
Dry        6589
Summer     2644
Deyr       2501
Gu         2476
Long       1986
Name: count, dtype: int64

crop_production_system
All (PS)           69708
none               51461
Rainfed (PS)       20846
irrigated           8363
Other               7789
Commercial (PS)     1838
Name: count, dtype: int64


# Feature engineering: which columns we keep and drop

In [15]:
features = ["country", "product", "season_name", "crop_production_system",
            "planting_month", "harvest_month", "harvest_year", "area"]
target = "yield"

df = df[features + [target]].reset_index(drop=True)
print("Final shape:", df.shape)
print(df["country"].unique())
print("Rwanda rows:", (df["country"] == "Rwanda").sum())
df.head()

Final shape: (160005, 9)
<StringArray>
[                      'Angola',                 'Burkina Faso',
                      'Burundi',                        'Benin',
                          'DRC',     'Central African Republic',
                     'Cameroon',                     'Ethiopia',
                        'Ghana',                        'Kenya',
                      'Liberia',                      'Lesotho',
                   'Madagascar',                         'Mali',
                   'Mauritania',                       'Malawi',
                   'Mozambique',                        'Niger',
                      'Nigeria',                       'Rwanda',
                        'Sudan',                      'Senegal',
                      'Somalia',                  'South Sudan',
                         'Chad',                         'Togo',
 'Tanzania, United Republic of',                       'Uganda',
                 'South Africa',                   

,country,product,season_name,crop_production_system,planting_month,harvest_month,harvest_year,area,yield
0,Angola,Banana,Main,All (PS),10,4,2008,1364.0,14.360704
1,Angola,Banana,Main,All (PS),10,4,2009,1364.0,15.000733
2,Angola,Banana,Main,All (PS),10,4,2015,8706.0,25.504709
3,Angola,Banana,Main,All (PS),10,4,2016,8706.0,25.504709
4,Angola,Banana,Main,All (PS),10,4,2017,9549.7,24.499932
